# F67 — INE BGRI 2021 → Google Drive

Downloads the exact official INE **BGRI21_CONT.zip** archive, validates the ZIP and expected `BGRI21_CONT.gpkg` member, computes SHA-256, and copies the verified archive to Google Drive.

**Source identity is fixed:** no mirror, GRID, CAOP, municipal subset, or third-party geometry is substituted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib, shutil, zipfile, requests, time

URL = 'https://mapas.ine.pt/download/filesGPG/2021/BGRI21_CONT.zip'
TMP = Path('/content/BGRI21_CONT.zip')
DEST_DIR = Path('/content/drive/MyDrive/Mobilidade_Norte/F67')
DEST = DEST_DIR / 'BGRI21_CONT.zip'
MIN_BYTES = 200_000_000
EXPECTED_MEMBER = 'BGRI21_CONT.gpkg'

DEST_DIR.mkdir(parents=True, exist_ok=True)

headers = {'User-Agent': 'Mozilla/5.0 MobilidadeNorte-F67-Colab/1.0'}
with requests.get(URL, headers=headers, stream=True, timeout=(60, 1200), allow_redirects=True) as r:
    r.raise_for_status()
    total = int(r.headers.get('content-length') or 0)
    done = 0
    last_report = 0
    with TMP.open('wb') as f:
        for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
            if not chunk:
                continue
            f.write(chunk)
            done += len(chunk)
            if done - last_report >= 25 * 1024 * 1024:
                if total:
                    print(f'{done/1024**2:.1f} MiB / {total/1024**2:.1f} MiB')
                else:
                    print(f'{done/1024**2:.1f} MiB')
                last_report = done

size = TMP.stat().st_size
assert size >= MIN_BYTES, f'Archive unexpectedly small: {size:,} bytes'
assert zipfile.is_zipfile(TMP), 'Downloaded object is not a valid ZIP archive'

with zipfile.ZipFile(TMP) as z:
    names = z.namelist()
    basenames = {Path(n).name for n in names if not n.endswith('/')}
    assert EXPECTED_MEMBER in basenames, f'Missing expected {EXPECTED_MEMBER}'
    gpkg_entries = [n for n in names if Path(n).name == EXPECTED_MEMBER]

h = hashlib.sha256()
with TMP.open('rb') as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
        h.update(block)
digest = h.hexdigest()

shutil.copy2(TMP, DEST)
assert DEST.stat().st_size == size

print('\nPASS — exact F67 producer archive acquired and validated')
print(f'Size: {size:,} bytes ({size/1024**2:.2f} MiB)')
print(f'SHA-256: {digest}')
print(f'Expected GeoPackage member: {gpkg_entries[0]}')
print(f'Drive path: {DEST}')


After the last cell prints `PASS`, share the resulting `BGRI21_CONT.zip` from Drive with ChatGPT (or provide its Drive link). The private ingestion step should verify the same byte length/SHA-256 before preservation.